In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
openai_client


# Q 1


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
#files 

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:



In [6]:
len(documents)

72

In [7]:
documents[3]

{'content': '# The Course FAQ Dataset\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=Mx6EqvzVDz0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we build the RAG pipeline, let\'s get familiar with the data\nwe\'ll use as our knowledge base.\n\nWe run these courses every year, and students keep asking the same\nquestions in Slack. We collected those into an FAQ so people can find\nanswers before asking. Some courses have run for five cohorts, so the\nFAQ grows large and searching it by hand gets tedious. That\'s exactly\nthe problem our RAG system will solve.\n\nThe FAQ data is available as JSON from the DataTalks.Club website. I\nmaintain that site, so I made the data available at a JSON endpoint we\ncan fetch directly.\n\nLet\'s fetch it:\n\n```python\nimport requests\n\ndocs_url = "https://datatalks.club/faq/json/courses.json"\nresponse = requests.get(docs_url)\ncourses_raw = response.json()\n```\n\nThis returns a list of courses. Each course has a `path` field that

Q1. How many lesson pages
How many lesson pages are in the dataset?

24
## 72
240

720

# Q2. Indexing and searching

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

01-agentic-rag/lessons/03-rag.md

# 01-agentic-rag/lessons/14-agentic-loop.md

04-evaluation/lessons/13-llm-as-judge.md

06-best-practices/lessons/02-hybrid-search.md

In [8]:
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

# How does the agentic loop keep calling the model until it stops?

In [9]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [10]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [11]:
search_results = index.search(
    question,
    
    num_results=5
)

# search_results

In [12]:
type(search_results), len(search_results), type(search_results[0])

(list, 5, dict)

In [13]:
search_results[0].keys()

dict_keys(['content', 'filename'])

In [14]:
search_results[0]['filename']


'01-agentic-rag/lessons/14-agentic-loop.md'

# Q3. RAG


Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

700

# 7000

70000

700000

We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).



the helper as having 4 stages:

1. Search
2. Build Context
3. Build Prompt
4. Call GPT

In [15]:
from rag_helper import RAGBase

In [16]:
rag = RAGBase(
    index=index,
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

In [17]:
answer, usage = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)




In [18]:
print(answer)


It keeps calling the model in a `while True` loop.

Each turn it:
1. sends the full message history to the model,
2. checks the response for any `function_call` items,
3. runs those tools and appends the tool outputs to `messages`,
4. repeats.

It stops when a response comes back with no function calls:

```python
if has_function_calls == False:
    break
```

So the exit condition is: no tool calls this turn, meaning the model has produced its final answer.


In [19]:
print(usage)

ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7238)


# Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With size=2000 and step=1000 (you can see the implementation here):


Each chunk is a window of size characters of the page.

The window moves forward by step characters between chunks. Since step is smaller than size, consecutive chunks overlap by size - step (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.

Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).

How many chunks do you get?

70

 # 295

1100

4500


In [20]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


# Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

about the same

# 3× fewer

10× fewer

30× fewer

In [22]:
chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

chunk_index.fit(chunks)

In [23]:
rag_chunks = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

In [24]:
answer, usage = rag_chunks.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(usage)

ResponseUsage(input_tokens=2309, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2399)


In [25]:
input_tokens = 7126
input_tokens_2 = 2309

In [26]:
7126 / 2309

3.086184495452577

# Q6. Turning it into an agent
So far search runs once, with the exact query. Let's make it agentic: give the LLM a search tool and let it decide when (and what) to search. We suggest toyaikit, the small agent library from the module, but you can use anything you like - the OpenAI Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

uv add toyaikit

Create a search function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your search tool and run it (with toyaikit, the same way as in the ToyAIKit lesson). Use these instructions for the agent (they nudge it to search a few times):

You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many times it called the search tool.

How many times did the agent call search?

Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

0

# 4

10

20


In [27]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner

In [31]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

In [32]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

### Step 1 - Create the tool using your chunk_index

In [33]:
search_count = 0

def search(query: str) -> list:
    """
    Search the lesson chunks.
    """
    global search_count

    search_count += 1

    return chunk_index.search(
        query,
        num_results=5
    )

### Step 2 - Register the tool

In [34]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

### Step 3 - Create the instructions

In [35]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

### Step 4 - Create the runner

In [36]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

### Step 5 - Run the exact  question

In [37]:
search_count = 0

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [38]:
print(search_count)

3
